#  OLIST 360 Business Intelligence Analysis
## Notebook 01 — Data Loading & Cleaning
**Goal:** Load all 9 datasets, explore, clean and merge into one master dataframe  
**Tools:** Python, Pandas, NumPy  
**Data:** Brazilian E-Commerce Public Dataset by Olist — 100,000+ real orders

In [1]:

import pandas as pd
import numpy as np
import os                        
import warnings
warnings.filterwarnings('ignore') 

print("ALL LIBRARIES IMPORTED SUCCESFULLY")

ALL LIBRARIES IMPORTED SUCCESFULLY


In [2]:

BASE_PATH ="../data/raw/"



orders = pd.read_csv(BASE_PATH + "olist_orders_dataset.csv")
customers = pd.read_csv(BASE_PATH + "olist_customers_dataset.csv")
order_items = pd.read_csv(BASE_PATH + "olist_order_items_dataset.csv")
products = pd.read_csv(BASE_PATH + "olist_products_dataset.csv")
sellers = pd.read_csv(BASE_PATH + "olist_sellers_dataset.csv")
payments = pd.read_csv(BASE_PATH + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(BASE_PATH + "olist_order_reviews_dataset.csv")
geolocation = pd.read_csv(BASE_PATH + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(BASE_PATH + "product_category_name_translation.csv")

print("ALL 9 DATASETS LOADED SUCCESFULLY")

ALL 9 DATASETS LOADED SUCCESFULLY


In [3]:

datasets = {
    "orders": orders,
    "customers": customers,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
    "payments": payments,
    "reviews": reviews,
    "geolocation": geolocation,
    "category_translation": category_translation
}

print("Dataset shapes( rows x columns):\n")
for name, df in datasets.items():
    print(f" {name:25s} ->{df.shape[0]:,} rows x {df.shape[1]} columns")


Dataset shapes( rows x columns):

 orders                    ->99,441 rows x 8 columns
 customers                 ->99,441 rows x 5 columns
 order_items               ->112,650 rows x 7 columns
 products                  ->32,951 rows x 9 columns
 sellers                   ->3,095 rows x 4 columns
 payments                  ->103,886 rows x 5 columns
 reviews                   ->99,224 rows x 7 columns
 geolocation               ->1,000,163 rows x 5 columns
 category_translation      ->71 rows x 2 columns


## MISSING VALUES

In [4]:
print("Missing values summary")
for name, df in datasets.items():
    missing=df.isnull().sum().sum()
    pct = (missing / (df.shape[0] * df.shape[1])) * 100 
    print(f" {name:25s} -> {missing:,} missing_values({pct:.2f}%)")

Missing values summary
 orders                    -> 4,908 missing_values(0.62%)
 customers                 -> 0 missing_values(0.00%)
 order_items               -> 0 missing_values(0.00%)
 products                  -> 2,448 missing_values(0.83%)
 sellers                   -> 0 missing_values(0.00%)
 payments                  -> 0 missing_values(0.00%)
 reviews                   -> 145,903 missing_values(21.01%)
 geolocation               -> 0 missing_values(0.00%)
 category_translation      -> 0 missing_values(0.00%)


In [5]:
print("ORDERS - Column wise missing values:\n")

print(orders.isnull().sum())
print(f"\n Sample data:")
orders.head(3)

ORDERS - Column wise missing values:

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

 Sample data:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


## Check data type

In [6]:
print("Orders Data type")
print(orders.dtypes)

Orders Data type
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


## Converting all datecolumns to datetime

In [7]:
date_columns =[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col],errors='coerce')

    print("Datetime conversion done:")
    print("\n Updated data types:")
    print(orders[date_columns].dtypes)

Datetime conversion done:

 Updated data types:
order_purchase_timestamp         datetime64[us]
order_approved_at                           str
order_delivered_carrier_date                str
order_delivered_customer_date               str
order_estimated_delivery_date               str
dtype: object
Datetime conversion done:

 Updated data types:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date                str
order_delivered_customer_date               str
order_estimated_delivery_date               str
dtype: object
Datetime conversion done:

 Updated data types:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date               str
order_estimated_delivery_date               str
dtype: object
Datetime conversion done:

 Updated data types:
order_purchase_timestamp         datetime64[us

## HANDLING MISSING VALUES IN ORDERS

In [8]:
before =len(orders)
orders_clean = orders.dropna(subset=['order_id','customer_id'])
after =len(orders_clean)

print(f" Orders before cleaning : {before:,}")
print(f" Orders after cleaning : {after:,}")
print(f" Rows Dropped :{before - after}")
print(f"\n Missing delivery dates KEPT - They are valid in - progress orders")

 Orders before cleaning : 99,441
 Orders after cleaning : 99,441
 Rows Dropped :0

 Missing delivery dates KEPT - They are valid in - progress orders


## FIX PRODUCT DATASET

In [9]:
products_clean = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

products_clean['product_category_name_english']=(
    products_clean['product_category_name_english']
    .fillna('Unknown')
)

print(f"Products Cleaned")
print(f" Sample categories:")
print(products_clean['product_category_name_english'].value_counts().head(10))

Products Cleaned
 Sample categories:
product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64


## MASTER DATASET MERGE ALL DATASETS

In [10]:
master = orders_clean.merge(customers, on='customer_id', how='left')


master = master.merge(order_items, on ='order_id', how ='left')


master = master.merge(
    products_clean[['product_id','product_category_name_english',
                    'product_weight_g','product_length_cm']],
    on='product_id', how='left')


master = master.merge(sellers, on='seller_id', how='left')


master = master.merge(
    payments[['order_id','payment_type','payment_installments','payment_value']],
    on='order_id',how='left'
)

master = master.merge(
    reviews[['order_id','review_score']],
    on='order_id', how='left'
)

print(f" Master dataframe created")
print(f" Shape: {master.shape[0]:,} rows x {master.shape[1]} columns")
print(f"\n Columns in masters:")
print(list(master.columns))

 Master dataframe created
 Shape: 119,143 rows x 28 columns

 Columns in masters:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name_english', 'product_weight_g', 'product_length_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'payment_type', 'payment_installments', 'payment_value', 'review_score']


##  TO SAVE CLEANED MASTER DATA

In [11]:
master.to_csv("../data/processed/master_clean.csv", index=False)

print("Master dataframe saved to data/processed/")
print(f" Shape Saved: {master.shape[0]:,} rows x {master.shape[1]} columns")

Master dataframe saved to data/processed/
 Shape Saved: 119,143 rows x 28 columns
